In [1]:
# import packages needed throughout
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns

# Data Preprocessing

## Data Merging

In [2]:
from data_prep import unzip_data, merge_data

In [3]:
# Run only if original OULAD data is needed
# Unzips OULAD data and loads each csv file
zip_file = "Data/OULAD.zip" # "C:/Users/jacma/OneDrive/school work/UMD/Data Science Boot Camp/Group Project/Data/OULAD.zip"
assessments, courses, studentAssessments, studentInfo, studentRegistration, studentVLE, VLEdata = unzip_data(zip_file)

In [4]:
# Run to create csv with merged data (merges data from separate OULAD csv files) 
zip_file = "Data/OULAD.zip" # "C:/Users/jacma/OneDrive/school work/UMD/Data Science Boot Camp/Group Project/Data/OULAD.zip"
merged_file = "Data/merged_data.csv" # "C:/Users/jacma/OneDrive/school work/UMD/Data Science Boot Camp/Group Project/Data/merged_data.csv"
df = merge_data(zip_file, merged_file)

✅ Data partitioned by code_module & code_presentation and written into parquet files stored in Data/parquet_files
✅ Aggregated data partitioned by code_module & code_presentation and written into parquet files stored in Data/aggregated


AttributeError: 'DataFrame' object has no attribute 'data_registration'

In [ ]:
from IPython.display import Image
image_path = "data merging.png" # "C:/Users/jacma/OneDrive/school work/UMD/Data Science Boot Camp/Group Project/data merging.png"
Image(filename=image_path, width=800, height=800)

## Feature Calculation and Data Selection

In [ ]:
# run only if necessary - may already have selected data with calculated features
df = pd.read_csv("Data/merged_data.csv") # pd.read_csv("C:/Users/jacma/OneDrive/school work/UMD/Data Science Boot Camp/Group Project/Data/merged_data.csv")

In [ ]:
# from feature_calculator_old import feature_calculator
# df = feature_calculator(df,features=['submission','total','focus','regularity','diversity','demographics'],
#                         n_weeks=3,write_to=None)

In [4]:
from feature_calculator import vle_features, assessment_features
from combine_tables import combine_tables

# Generate engineered features on vle interaction data
df_vle_early = vle_features(df,features=['total','focus','regularity','diversity'], 
                            n_weeks=3,write_to=None)

# Generate engineered features on assessment data
# Requires running second code block in Data Merging that loads assessments
df_assessments_early = assessment_features(df,assessments,features=['submission'], 
                                           n_weeks=3,write_to=None)

# Combine vle & assessment data tables
df_early = combine_tables(df_vle_early,df_assessments_early)

# Add demographics data 
demographics_cols = ['gender','region','highest_education','imd_band', 'age_band',
                     'num_of_prev_attempts','studied_credits', 'disability']
df_demographics = df[['id_student','code_module','code_presentation'] + demographics_cols].drop_duplicates()
df_early = combine_tables(df_early,df_demographics)

# Modeling Procedures

## Define models

In [ ]:
from sklearn.model_selection import train_test_split

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier
from xgboost.sklearn import XGBClassifier
from sklearn.dummy import DummyClassifier

from sklearn.preprocessing import StandardScaler, RobustScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.base import clone
from sklearn.metrics import confusion_matrix, roc_auc_score, precision_score, recall_score, f1_score, accuracy_score
from sklearn.model_selection import StratifiedKFold

In [ ]:
# Define preprocessors
scale_cols = ['content_focus_pre_w3', 'collaborative_focus_pre_w3', 'active_days_per_week_pre_w3',
            'std_regularity_pre_w3', 'vle_richness_pre_w3', 'diversity_shannon_pre_w3', 
            'total_vle_pre_w3']
cat_cols = ['code_module', 'submission_type']

cat_processor = ColumnTransformer(
    transformers = [('cat_encode', OneHotEncoder(drop='first'), cat_cols)],
    remainder = 'passthrough'
)

catscale_processor = ColumnTransformer(
    transformers = [('scale', RobustScaler(), scale_cols),
                ('cat_encode', OneHotEncoder(drop='first'), cat_cols)],
    remainder = 'passthrough'
)

# Define model pipelines
models = {
    'Baseline (Stratified)': DummyClassifier(strategy='stratified', random_state=SEED),
    'Logistic Regression': Pipeline([('preprocess', catscale_processor), 
                                    ('classify', LogisticRegression(random_state=SEED, max_iter=5000, solver='saga',
                                                                    penalty='elasticnet', l1_ratio=0.5, C=0.1,
                                                                    class_weight='balanced'))]),
    'Random Forest': Pipeline([('preprocess', cat_processor), 
                            ('classify', RandomForestClassifier(random_state=SEED, n_estimators=100, max_depth=6,
                                                                min_samples_split=50, min_samples_leaf=25,
                                                                max_features='sqrt', max_samples=0.7,
                                                                class_weight='balanced'))]),
    'Gradient Boosting': Pipeline([('preprocess', cat_processor),
                                ('classify', GradientBoostingClassifier(random_state=SEED, n_estimators=100, 
                                                                        max_depth=3, learning_rate=0.1))]),
    'Extra Trees': Pipeline([('preprocess', cat_processor),
                            ('classify', ExtraTreesClassifier(random_state=SEED, n_estimators=100, max_depth=6,
                                                            min_samples_split=50, min_samples_leaf=25,
                                                            max_features='sqrt', class_weight='balanced'))]),
    'XGBoost': Pipeline([('preprocess', cat_processor),
                        ('classify', XGBClassifier(random_state=SEED))])
}

## Fit and tune models

In [ ]:
del df
from modeler import data_preprocess, model_traintest_split, model_initfit, model_tune

In [ ]:
# Preprocess data, use cross-validation to examine performances for various models
df_model_clean = model_preprocess(df)
results_df = model_initfit(df_model_clean, models)

# Display results
print("\nMODEL PERFORMANCE (Cross-Validation)")
print("-" * 80)
print(results_df.to_string(index=False, float_format=lambda x: f'{x:.4f}'))

print("\n" + "=" * 80)
print("KEY INSIGHTS")
print("=" * 80)

# Get baseline and best model
baseline = results_df[results_df['Model'] == 'Baseline (Stratified)'].iloc[0]
best = results_df.iloc[0]

print(f"\n✓ BEST ROC-AUC: {best['Model']}")
print(f"  • ROC-AUC: {best['Test ROC-AUC']:.3f} (+{best['Test ROC-AUC'] - baseline['Test ROC-AUC']:.3f} vs baseline)")
print(f"  • Recall: {best['Test Recall']:.1%} of at-risk students identified")
print(f"  • Precision: {best['Test Precision']:.1%}")
print(f"  • Overfitting: {best['Overfitting Gap']:.3f} {'✓ Well-controlled' if best['Overfitting Gap'] < 0.10 else '⚠ Moderate'}")

print(f"\n✓ RUNNER-UP ROC-AUC: {results_df.iloc[1]['Model']}")
runner_up = results_df.iloc[1]
print(f"  • ROC-AUC: {runner_up['Test ROC-AUC']:.3f}")
print(f"  • Recall: {runner_up['Test Recall']:.1%} of at-risk students identified")
print(f"  • Precision: {runner_up['Test Precision']:.1%}")
print(f"  • Overfitting: {runner_up['Overfitting Gap']:.3f} {'✓ Excellent' if abs(runner_up['Overfitting Gap']) < 0.05 else '✓ Good'}")



In [ ]:
# Hyperparameter tuning
final_models = model_tune(df_model_clean, models)


## Further analysis

In [ ]:
from analysis import metrics_at_thresholds, visualize_thresholds

In [ ]:
# Threshold analysis
thresholds = [0.3, 0.35, 0.4, 0.45, 0.5, 0.55]
tuned_models = [('LR', final_models['Logistic Regression']), 
                ('RF', final_models['Random Forest']), 
                ('ET', final_models['Extra Trees'])]

results = metrics_at_thresholds(thresholds, tuned_models, X_train, y_train)
df_results = pd.DataFrame(results)
visualize_thresholds(df_results)

print(df_results.to_string(index=False))

In [ ]:
# Feature Importance Analysis
cat_features = list(tuned_models['LR'].named_steps['preprocess'].
                    named_transformers_['cat_encode'].get_feature_names_out(cat_cols))
all_features = scale_cols + cat_features

# Extract importances
importances = {
    'LR': pd.DataFrame({
        'Feature': all_features,
        'Value': tuned_models['LR'].named_steps['classify'].coef_[0]
    }).sort_values('Value', key=abs, ascending=False),
    'RF': pd.DataFrame({
        'Feature': cat_features + scale_cols,
        'Value': tuned_models['RF'].named_steps['classify'].feature_importances_
    }).sort_values('Value', ascending=False),
    'ET': pd.DataFrame({
        'Feature': cat_features + scale_cols,
        'Value': tuned_models['ET'].named_steps['classify'].feature_importances_
    }).sort_values('Value', ascending=False)
}

# Display top 10
for model, df in importances.items():
    print(f"\n{model} - Top 10 Features")
    print("=" * 60)
    print(df.head(10).to_string(index=False))

# Visualization
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
colors = {'LR': ['red' if v < 0 else 'blue' for v in importances['LR'].head(10)['Value']], 
          'RF': 'green', 'ET': 'orange'}

for ax, (model, df) in zip(axes, importances.items()):
    top = df.head(10)
    ax.barh(range(len(top)), top['Value'], color=colors[model], alpha=0.7)
    ax.set_yticks(range(len(top)))
    ax.set_yticklabels(top['Feature'])
    if model == 'LR':
        ax.axvline(x=0, color='black', linestyle='-', linewidth=1)
    ax.set_xlabel('Coefficient' if model == 'LR' else 'Importance')
    ax.set_title(f'{model} Feature Importance', fontweight='bold')
    ax.invert_yaxis()
    ax.grid(alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

# Test set performance

In [ ]:
# Test Set Evaluation at Threshold 0.35
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

y_test_numeric = y_test if y_test.dtype != 'object' else y_test.apply(lambda x: int(x[0]))
models_eval = [('LR', tuned_lr_model), ('RF', tuned_rf_model)]

# Calculate metrics
results = metrics_at_thresholds[[0.35],models_eval]

print("TEST SET PERFORMANCE (Threshold = 0.35)")
print("=" * 60)
print(pd.DataFrame(results).to_string(index=False, float_format=lambda x: f'{x:.4f}'))

# Confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for idx, (name, model) in enumerate(models_eval):
    y_pred = (model.predict_proba(X_test)[:, 1] >= 0.35).astype(int)
    ConfusionMatrixDisplay(confusion_matrix(y_test_numeric, y_pred), 
                          display_labels=['Pass', 'At-Risk']).plot(cmap='Blues', ax=axes[idx], values_format='d')
    axes[idx].set_title(f'{name} (Threshold=0.35)', fontweight='bold')
plt.tight_layout()
plt.show()